In [ ]:
import numpy as np
import scipy.stats as st
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az
import pytensor
from utils import create_bipartite_bayesian_network_cond, create_bipartite_bayesian_network_nocond
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def f_2(x):
            return (1 - x) ** 4

def f_3(x):
    term1 = 1 / (x * np.sqrt(3 * np.pi))
    exponent = -0.5 * ((np.log(x) + np.sqrt(1.5) - np.log(3)) ** 2)
    return term1 * np.exp(exponent)

def f_4(x):
    return f_3(1 - x)

def f_5(x):
    return f_2(1 - x)

In [ ]:
def generate_data(n_students, m_items, random_state=42):
    """
    params:
        n_studetns (int): количество студентов
        m_items (int): количество предметов
    """
    np.random.seed(random_state)
    return np.random.randint(low=2, high=5, size=n_students * m_items).reshape(n_students, m_items)

In [ ]:
def generate_rating_matrix(n_students, m_items, seed=42):
    """
    params:
        n_studetns (int): количество студентов
        m_items (int): количество предметов
    """
    np.random.seed(seed)

    perf = st.beta(a=2, b=2).rvs(size=n_students)
    diff = st.beta(a=2, b=2).rvs(size=m_items)

    ratio = np.zeros((n_students, m_items))

    rating_matrix = np.zeros_like(ratio)
    
    for i, perf_i in enumerate(perf):
        for j, diff_j in enumerate(diff):
            ratio[i, j] = perf_i / (perf_i + diff_j)

            w2 = f_2(ratio[i, j])
            w3 = f_3(ratio[i, j])
            w4 = f_4(ratio[i, j])
            w5 = f_5(ratio[i, j])
            w_summ = w2 + w3 + w4 + w5

            p2 = w2 / w_summ
            p3 = w3 / w_summ
            p4 = w4 / w_summ
            p5 = w5 / w_summ

            marks = np.random.multinomial(n=1, pvals=[p2, p3, p4, p5], size=1).flatten()

            def eval_mark(marks):
                for i, mark in enumerate(marks):
                    if mark == 1:
                        return i + 2
                    
            rating_matrix[i, j] = eval_mark(marks)

    return rating_matrix, perf, diff


In [ ]:
def eval_stats(rm_shapes, model, model_params):
    """
    params:
        rm_shapes (list((n1, m1), ..., (nk, mk)): shape'ы матриц рейтинга
        model: модель
        model_params dict(): параметры модели
    """
    mae_stats = []
    max_error_stats = []
    for rm_shape in rm_shapes:
        n_students, m_items = rm_shape
        rm, perf, diff = generate_rating_matrix(n_students, m_items)

        trace, _ = model(ratings_matrix=rm, **model_params)

        mae = np.mean(np.abs(az.summary(trace)['mean'] - np.concatenate([perf, diff], axis=0)))
        max_error = np.max(np.abs(az.summary(trace)['mean'] - np.concatenate([perf, diff], axis=0)))

        mae_stats.append(mae)
        max_error_stats.append(max_error)

    return mae_stats, max_error_stats

In [ ]:
rm_shapes = [(5, 5), (10, 10), (20, 20)]
params = {
    'student_alpha': 2,
    'student_beta': 2,
    'item_alpha': 2,
    'item_beta': 2,
    'draws':1000,
    'tune':2000,
    'chains':6,
    'cores':6,

}

cond_stats = eval_stats(rm_shapes, create_bipartite_bayesian_network_cond, model_params=params)

Sampling 6 chains for 2_000 tune and 1_000 draw iterations (12_000 + 6_000 draws total) took 71 seconds.
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (6 chains in 6 jobs)
NUTS: [student_ability, item_difficulty]


Sampling 6 chains for 2_000 tune and 1_000 draw iterations (12_000 + 6_000 draws total) took 76 seconds.
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (6 chains in 6 jobs)
NUTS: [student_ability, item_difficulty]


In [ ]:
rm_shapes = [(5, 5), (10, 10), (20, 20)]
params = {
    'student_alpha': 2,
    'student_beta': 2,
    'item_alpha': 2,
    'item_beta': 2,
    'draws':1000,
    'tune':2000,
    'chains':6,
    'cores':6
}

no_cond_stats = eval_stats(rm_shapes, create_bipartite_bayesian_network_nocond, model_params=params)

In [ ]:
print("Статистики для модели с условием на отличников и двоечников:")
print(cond_stats)
print("Статистики для модели без условия:")
print(no_cond_stats)

In [ ]:
rm, perf, diff = generate_rating_matrix(10, 5)

trace, model = create_bipartite_bayesian_network_cond(ratings_matrix=rm, **params)

In [ ]:
# смотрим как модель отображает красиво зависимости

pm.model_to_graphviz(model)

In [ ]:
# смотрим на граф вычислений (зависимости, которые учитываются в pyMC)

pytensor.dprint(model.logp())

In [ ]:
"""
MAE реализации сл величины от среднего оценки
"""
np.mean(np.abs(az.summary(trace)['mean'] - np.concatenate([perf, diff], axis=0)))

In [ ]:
"""
Максимальное отклонение реализации сл величины от среднего оценки
"""
np.max(np.abs(az.summary(trace)['mean'] - np.concatenate([perf, diff], axis=0)))

In [ ]:
def show_trace(trace, figsize):
    ncols, nrows = figsize
    fig, axes = plt.subplots(ncols=ncols, nrows=nrows, figsize=(14, 30))
    
    az.plot_trace(trace,
                compact=False,
                legend=True,
                axes=axes)
    axes = axes.flatten()
    for i in range(0, len(axes), 2):
        axes[i].set_xlim(0, 1)
    return fig, axes

In [ ]:
fig, axes = show_trace(trace, (2, 15))
rvals = np.concatenate([perf, diff], axis=0)
for i in range(0, len(rvals)):
    axes[i * 2].axvline(rvals[i])
plt.tight_layout()
plt.show()